# The Blockbuster Formula
## Can money buy a great movie?
**IS 445 Final Project Part 1 — Naman Behl (individual)**

Exploratory draft for review. The aim is to distinguish commercial performance from audience reception.

## Dataset provenance, access and license
**Name:** Vega Movies (movies.json). Distributed by the Vega visualization project as a teaching dataset, also used in the University of Washington Interactive Data Lab visualization curriculum. It combines film financial and ratings data from multiple sources; the curriculum warns of uneven sampling and errors. Original collection methods and inclusion criteria are not fully documented, so this is not a representative census of cinema.

[Catalog and field documentation](https://vega.github.io/vega-datasets/datasets/movies/) | [Teaching documentation](https://idl.uw.edu/visualization-curriculum/altair-data-transformation/) | [Data URL](https://cdn.jsdelivr.net/npm/vega-datasets@1/data/movies.json)

**License:** The dataset catalog specifies no license for this file. The repository software license does not establish the rights to the underlying movie data. My best assessment is that attributed classroom analysis of this publicly distributed teaching dataset is consistent with its stated educational purpose, but unrestricted redistribution or commercial use cannot be assumed. For later public deliverables I will link to the original source, avoid republishing the full raw dataset, and verify permission for any publication of derived charts before proceeding.

In [ ]:
import io
from urllib.request import urlopen
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://cdn.jsdelivr.net/npm/vega-datasets@1/data/movies.json"
with urlopen(DATA_URL, timeout=60) as response:
    raw = response.read()
movies = pd.read_json(io.BytesIO(raw))
# Different Vega releases may use spaces or underscores in field names.
movies.columns = movies.columns.str.strip().str.replace(" ", "_", regex=False)
print("Rows:", movies.shape[0], "Columns:", movies.shape[1])
print("Downloaded bytes:", len(raw))
print("Memory (MiB):", movies.memory_usage(deep=True).sum()/1024**2)
display(movies.head())

## Dataset size and hosting
The catalog reports 3,201 records and 16 fields, with a JSON file of approximately 1.4 MB. The code above calculates actual bytes and in-memory size. This is below GitHub's 100 MiB file limit, but I plan to load from the upstream URL instead of redistributing the raw file. [GitHub limits](https://docs.github.com/en/repositories/working-with-files/managing-large-files/about-large-files-on-github).

In [ ]:
movies.info()
display(pd.DataFrame({"dtype":movies.dtypes.astype(str), "missing":movies.isna().sum(), "missing_pct":movies.isna().mean().mul(100).round(2), "distinct":movies.nunique()}))
display(movies.describe(include="all").T)
print("Exact duplicates:", movies.duplicated().sum())

## Fields and measurement
Each row is a movie record, not an individual viewer or ticket purchase. Title, distributor, director, major genre, source, creative type and MPAA rating are categorical labels (MPAA rating is an age classification). Release date is temporal. US gross, worldwide gross, US DVD sales and production budget are quantitative monetary amounts; runtime is quantitative in minutes; IMDb and Rotten Tomatoes ratings are numerical reception measures and IMDb votes are counts. Worldwide gross includes domestic gross, so adding them would double-count revenue. Ratings are proxies for reception and not objective measures of film quality.

In [ ]:
df = movies.drop_duplicates().copy()
df["Title"] = df["Title"].astype("string")
df["Release_Date"] = pd.to_datetime(df["Release_Date"], format="%b %d %Y", errors="coerce")
df["Year"] = df["Release_Date"].dt.year
for col in ["Production_Budget","Worldwide_Gross","IMDB_Rating","Rotten_Tomatoes_Rating","IMDB_Votes"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("Unparsed or missing dates:", df["Release_Date"].isna().sum())
print("Nonpositive budgets:", df["Production_Budget"].le(0).sum())
print("Nonpositive grosses:", df["Worldwide_Gross"].le(0).sum())
display(df["Year"].describe())
display(df["Major_Genre"].value_counts(dropna=False))

In [ ]:
finance = df.loc[df["Production_Budget"].gt(0) & df["Worldwide_Gross"].gt(0)].copy()
finance["Gross_Multiple"] = finance["Worldwide_Gross"] / finance["Production_Budget"]
quality = df.loc[df["Production_Budget"].gt(0) & df["IMDB_Rating"].between(0,10)].copy()
quality["Log_Budget"] = np.log10(quality["Production_Budget"])
print("Financial sample:",len(finance),"Budget/rating sample:",len(quality))
print("Pearson correlation, log budget and IMDb rating:",quality[["Log_Budget","IMDB_Rating"]].corr().iloc[0,1])
print("Spearman rank correlation:",quality[["Production_Budget","IMDB_Rating"]].corr(method="spearman").iloc[0,1])
display(finance["Gross_Multiple"].describe(percentiles=[.25,.5,.75,.9,.99]))

## Main plot — Does spending more buy better audience reception?
Each dot represents a film with a positive recorded budget and a valid IMDb rating. A logarithmic budget axis allows small independent productions and large studio films to share the same chart. The relationship is interesting because it separates spending from viewer reception; an association, if present, would not prove that budget causes higher ratings.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(quality["Production_Budget"],quality["IMDB_Rating"],alpha=.3,s=14)
ax.set_xscale("log")
ax.set(xlabel="Production budget (USD, log scale)",ylabel="IMDb rating (0–10)",title="Can money buy a great movie?")
plt.show()

## Additional exploration
Compare spending to worldwide gross and examine gross revenue multiples by genre. A gross multiple is not profit or ROI: marketing, theatrical revenue shares, distribution and other costs are not observed. Monetary values are nominal and not adjusted for inflation.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(finance["Production_Budget"],finance["Worldwide_Gross"],alpha=.25,s=12)
ax.set(xscale="log",yscale="log",xlabel="Production budget (USD)",ylabel="Worldwide gross (USD)",title="Spending versus box office")
plt.show()
genres = finance.groupby("Major_Genre")["Gross_Multiple"].agg(["size","median"]).query("size >= 20").sort_values("median")
display(genres)
genres["median"].plot.barh(title="Median worldwide gross / budget by genre")
plt.xlabel("Gross revenue multiple")
plt.show()

In [ ]:
paired = df[df["IMDB_Rating"].between(0,10) & df["Rotten_Tomatoes_Rating"].between(0,100)].copy()
paired["Audience_100"] = paired["IMDB_Rating"] * 10
paired["Score_Gap"] = paired["Audience_100"] - paired["Rotten_Tomatoes_Rating"]
display(paired[["Title","Audience_100","Rotten_Tomatoes_Rating","Score_Gap"]].sort_values("Score_Gap",ascending=False).head(10))
print("Paired film count:",len(paired))

## Review and genuine debugging record
After running this notebook, record the actual memory measurement, the observed correlations, patterns and surprising films in your own words. Test a different budget threshold or vote-count threshold and compare the results. If a real attempt fails, preserve its code as comments and explain the error here; do not invent failed attempts. Note the limitations of missing values, inflation, possible source errors, selection bias and rating snapshots. Review dataset publication rights before Parts 2 and 3.